In [1]:
from pathlib import Path
import sys

import pandas as pd

HERE = Path.cwd().resolve()

PYTHON_ROOT = next(
    path
    for path in (HERE, *HERE.parents)
    if (path / "trading_portfolio").is_dir()
    and (path / "t212_universe").is_dir()
)

PROJECT_DIR = (
    PYTHON_ROOT
    / "trading_portfolio"
    / "uk_portfolio"
    / "equity_momentum"
)

UNIVERSE_DIR = PYTHON_ROOT / "t212_universe"

source_path = str(PROJECT_DIR / "src")
if source_path not in sys.path:
    sys.path.insert(0, source_path)

BASELINE = {
    "formation_sessions": 252,
    "skip_sessions": 21,
    "top_frac": 0.20,
    "rebalance_frequency": "monthly",
    "initial_capital_gbp": 10_000.0,
}

display(pd.Series(BASELINE, name="Baseline").to_frame())

,Baseline
formation_sessions,252
skip_sessions,21
top_frac,0.2
rebalance_frequency,monthly
initial_capital_gbp,10000.0


In [2]:
import importlib
import momentum_data

importlib.reload(momentum_data)

universe = momentum_data.load_candidate_universe(
    UNIVERSE_DIR,
    PROJECT_DIR / "data" / "candidate_universe_v1.csv",
)

print(f"Frozen share candidates: {len(universe):,}")

display(
    universe[
        ["yf_symbol", "name", "isin", "quote_unit", "exchange"]
    ].head(10)
)

Frozen share candidates: 669


/Users/oscarlewis/Desktop/python/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


,yf_symbol,name,isin,quote_unit,exchange
0,88E.L,88 Energy,AU00000088E2,GBX,London Stock Exchange AIM
1,AURA.L,Aura Energy,AU000000AEE7,GBX,London Stock Exchange AIM
2,CLA.L,Celsius Resources,AU000000CLA6,GBX,London Stock Exchange AIM
3,EMH.L,European Metals Holdings,AU000000EMH5,GBX,London Stock Exchange AIM
4,GEO.L,Geo Exploration,AU000000GBP6,GBX,London Stock Exchange AIM
5,LIT.L,Litigation Capital Management,AU000000LCA6,GBX,London Stock Exchange AIM
6,SVML.L,Sovereign Metals,AU000000SVM6,GBX,London Stock Exchange AIM
7,WNX.L,Wellnex Life,AU0000162281,GBX,London Stock Exchange AIM
8,SYN.L,Synergia Energy,AU0000233538,GBX,London Stock Exchange AIM
9,ALL.L,Atlantic Lithium,AU0000237554,GBX,London Stock Exchange AIM


In [3]:
import momentum_preparation

importlib.reload(momentum_preparation)

# Builds from existing local caches once; later runs load the frozen snapshot.
prepared_data = momentum_preparation.prepare_momentum_data(PROJECT_DIR)

prices = prepared_data["prices"]
market = prepared_data["market"]
schedule = prepared_data["schedule"]
price_coverage = prepared_data["coverage"]

pd.testing.assert_frame_equal(universe, prepared_data["universe"])
display(pd.Series(prepared_data["manifest"]["summary"], name="Prepared data").to_frame())

,Prepared data
candidate_tickers,669
downloaded_tickers,593
inherited_tickers,573
additional_tickers,20
unavailable_tickers,76
sessions,2779
rows,1647947
first_session,2015-01-02
last_session,2025-12-31
observed_rows,1363849


In [4]:
# Availability is separate from trading eligibility, which comes next.
display(
    price_coverage.groupby("source").agg(
        candidates=("yf_symbol", "size"),
        usable_close_sessions=("usable_close_sessions", "sum"),
    )
)

display(
    price_coverage.loc[
        price_coverage["unit_quarantine_sessions"].gt(0),
        ["yf_symbol", "first_usable_close", "last_usable_close", "unit_quarantine_sessions"],
    ].set_index("yf_symbol")
)

,candidates,usable_close_sessions
source,,
additional_yahoo,20,45112
inherited_reconciled,573,1312540
unavailable,76,0


,first_usable_close,last_usable_close,unit_quarantine_sessions
yf_symbol,,,
SPDI.L,2015-01-02,2025-07-30,107
BAY.L,2021-09-30,2025-08-19,93
VOX.L,2022-10-31,2024-02-02,483


In [5]:
# These dated suspensions were added to the inherited suspension register.
display(
    pd.DataFrame(prepared_data["review"]["suspensions"])[
        ["ticker", "start", "first_suspended_open", "resume"]
    ]
)

# Raw downloads remain unchanged. No replacement prices or fills are invented.
audit_columns = ["invalid_price", "invalid_volume", "suspected_unit_jump"]
display(
    pd.DataFrame({
        "Before review": prepared_data["raw_audit"][audit_columns].sum(),
        "After quarantine": prepared_data["cleaned_audit"][audit_columns].sum(),
    })
)

,ticker,start,first_suspended_open,resume
0,BLU.L,2018-07-26,2018-07-26,2019-01-24
1,SCGL.L,2016-03-29,2016-03-29,2017-02-28
2,SCGL.L,2018-04-30,2018-04-30,2018-10-30
3,SPDI.L,2025-11-06,2025-11-06,None


,Before review,After quarantine
invalid_price,0,0
invalid_volume,0,0
suspected_unit_jump,45,0


In [6]:
unavailable_candidates = price_coverage.loc[
    ~price_coverage["has_prices"], ["yf_symbol", "name", "unavailable_reason"]
].copy()

print(f"Unavailable candidates retained in the catalogue: {len(unavailable_candidates):,}")
print("No signals, performance results, or development/holdout split have been built yet.")
print("Prices are daily reference observations; liquidity and execution checks still apply.")
display(unavailable_candidates.head(10))

Unavailable candidates retained in the catalogue: 76
No signals, performance results, or development/holdout split have been built yet.
Prices are daily reference observations; liquidity and execution checks still apply.


,yf_symbol,name,unavailable_reason
25,RQIH.L,R&Q Insurance,YFTzMissingError: $RQIH.L: possibly delisted; ...
29,FOG.L,Falcon Oil & Gas,YFTzMissingError: $FOG.L: possibly delisted; n...
69,SCE.L,Surface Transforms,YFTzMissingError: $SCE.L: possibly delisted; n...
73,UBG.L,Unbound Group,YFTzMissingError: $UBG.L: possibly delisted; n...
100,ZYT.L,Zytronic,YFTzMissingError: $ZYT.L: possibly delisted; n...
103,RNO.L,Renold,YFTzMissingError: $RNO.L: possibly delisted; n...
106,SIXH.L,600 Group,YFTzMissingError: $SIXH.L: possibly delisted; ...
141,AFRN.L,Aferian,YFTzMissingError: $AFRN.L: possibly delisted; ...
142,HRN.L,Hornby,YFTzMissingError: $HRN.L: possibly delisted; n...
144,FEN.L,Frenkel Topping,YFTzMissingError: $FEN.L: possibly delisted; n...


In [7]:
import json

PERIODS = {"warmup": {"start": "2015-01-01", "end_exclusive": "2016-01-01"},
    "development": {"start": "2016-01-01", "end_exclusive": "2022-01-01"},
    "validation": {"start": "2022-01-01", "end_exclusive": "2024-01-01"},
    "holdout": {"start": "2024-01-01", "end_exclusive": "2026-01-01"}}

period_masks = pd.DataFrame({name: ((schedule.index >= bounds["start"])& (schedule.index < bounds["end_exclusive"]))
        for name, bounds in PERIODS.items()}, index=schedule.index)

#every session belongs to exactly one period
assert period_masks.sum(axis=1).eq(1).all()
assert period_masks.any().all()

protocol_path = PROJECT_DIR / "data" / "research_periods_v1.json"
if protocol_path.exists():
    if json.loads(protocol_path.read_text()) != PERIODS:
        raise ValueError("Research periods differ from the saved protocol. Review the change before replacing it.")
else:
    protocol_path.write_text(json.dumps(PERIODS, indent=2) + "\n")

period_summary = pd.DataFrame([{
            "period": name,
            "first_session": schedule.index[mask][0],
            "last_session": schedule.index[mask][-1],
            "sessions": int(mask.sum()),
        } for name, mask in period_masks.items()]).set_index("period")

display(period_summary)

,first_session,last_session,sessions
period,,,
warmup,2015-01-02,2015-12-31,253
development,2016-01-04,2021-12-31,1518
validation,2022-01-04,2023-12-29,501
holdout,2024-01-02,2025-12-31,507


In [8]:
import momentum_features

importlib.reload(momentum_features)

LIQUIDITY = {
    "lookback_sessions": 60,
    "min_positive_volume_fraction": 0.95,
    "min_median_traded_value_gbp": 100_000.0,
}

liquidity = momentum_features.build_liquidity_features(
    prices,
    schedule,
    lookback=LIQUIDITY["lookback_sessions"],
)

liquidity["liquidity_eligible"] = (
    liquidity["liquidity_history_complete"]
    & liquidity["median_traded_value_gbp"].ge(
        LIQUIDITY["min_median_traded_value_gbp"]
    )
    & liquidity["positive_volume_fraction"].ge(
        LIQUIDITY["min_positive_volume_fraction"]
    )
    & prices["positive_reported_volume"].eq(True)
    & prices["close_reference_usable"].eq(True)
)

eligible_counts = (
    liquidity["liquidity_eligible"]
    .groupby(level="Date")
    .sum()
)

development_dates = schedule.index[period_masks["development"]]

display(
    eligible_counts.loc[development_dates]
    .describe()
    .to_frame("Stocks passing liquidity checks")
)

,Stocks passing liquidity checks
count,1518.000000
mean,99.069170
std,39.695875
min,25.000000
25%,74.000000
50%,91.000000
75%,113.000000
max,203.000000


In [9]:
importlib.reload(momentum_features)

raw_momentum = momentum_features.build_raw_momentum_features(
    prices,
    schedule,
    formation_sessions=BASELINE["formation_sessions"],
    skip_sessions=BASELINE["skip_sessions"],
)

features = liquidity.join(raw_momentum, validate="one_to_one")

features["eligible"] = (
    features["liquidity_eligible"]
    & features["formation_history_complete"]
    & features["raw_momentum_score"].notna()
)

example = features.xs(development_dates[0], level="Date")

display(
    example.loc[
        example["eligible"],
        [
            "formation_start_price",
            "formation_end_price",
            "raw_momentum_score",
        ],
    ].head(10)
)

,formation_start_price,formation_end_price,raw_momentum_score
ticker,,,
88E.L,0.178358,0.126695,-0.289659
AMS.L,1.241570,1.703810,0.372302
BME.L,2.920237,3.170768,0.085791
BUR.L,1.206931,1.867883,0.547630
CHRT.L,2.384601,4.121321,0.728306
CPX.L,0.028750,0.051750,0.800000
CRTX.L,0.378261,0.233043,-0.383908
DEBS.L,0.392500,0.340000,-0.133758
DOTD.L,0.299671,0.439547,0.466764


In [10]:
import momentum_backtest

importlib.reload(momentum_backtest)

rebalance_calendar = momentum_backtest.build_monthly_rebalance_calendar(
    schedule
)

development_rebalances = rebalance_calendar.loc[
    rebalance_calendar["execution_date"].ge(
        PERIODS["development"]["start"]
    )
    & rebalance_calendar["execution_date"].lt(
        PERIODS["development"]["end_exclusive"]
    )
].copy()

print(f"Development rebalances: {len(development_rebalances)}")
display(development_rebalances.head())

Development rebalances: 72


,execution_date
signal_date,
2015-12-31,2016-01-04
2016-01-29,2016-02-01
2016-02-29,2016-03-01
2016-03-31,2016-04-01
2016-04-29,2016-05-03


In [11]:
importlib.reload(momentum_backtest)

development_targets, selection_summary = (
    momentum_backtest.build_momentum_targets(
        features,
        development_rebalances,
        top_frac=BASELINE["top_frac"],
    )
)

display(selection_summary.head())

first_execution = development_targets.index[0]
first_weights = development_targets.loc[first_execution]

display(
    first_weights.loc[first_weights.gt(0)]
    .rename("Target weight")
    .to_frame()
)

,signal_date,eligible_stocks,selected_stocks,target_cash_weight
Date,,,,
2016-01-04,2015-12-31,51,11,0.0
2016-02-01,2016-01-29,49,10,0.0
2016-03-01,2016-02-29,48,10,0.0
2016-04-01,2016-03-31,53,11,0.0
2016-05-03,2016-04-29,52,11,0.0


,Target weight
ticker,
CHRT.L,0.090909
CPX.L,0.090909
FEVR.L,0.090909
HCM.L,0.090909
HPOW.L,0.090909
JET2.L,0.090909
JLP.L,0.090909
OPTI.L,0.090909
STAF.L,0.090909


In [12]:
importlib.reload(momentum_backtest)

EXECUTION = {"cost_per_side_bps": 10.0}
first_rebalance = momentum_backtest.rebalance_with_costs(
    holdings_gbp=pd.Series(0.0, index=development_targets.columns),
    cash_gbp=BASELINE["initial_capital_gbp"],
    target_weights=development_targets.iloc[0],
    cost_per_side_bps=EXECUTION["cost_per_side_bps"])

display(pd.Series({
        "Starting equity": first_rebalance["equity_before_gbp"],
        "Amount invested": first_rebalance["holdings_gbp"].sum(),
        "Trading costs": first_rebalance["total_cost_gbp"],
        "Remaining cash": first_rebalance["cash_gbp"],
        "Equity after costs": first_rebalance["equity_after_gbp"],
    }).round(2).to_frame("GBP"))

,GBP
Starting equity,10000.00
Amount invested,9990.01
Trading costs,9.99
Remaining cash,0.00
Equity after costs,9990.01


In [13]:
importlib.reload(momentum_backtest)

first_date = development_targets.index[0]
second_date = schedule.index[schedule.index.get_loc(first_date) + 1]

first_day = momentum_backtest.step_portfolio_day(
    market_day=market.xs(first_date, level="Date"),
    units=pd.Series(0.0, index=development_targets.columns),
    cash_gbp=BASELINE["initial_capital_gbp"],
    target_weights=development_targets.loc[first_date],
    cost_per_side_bps=EXECUTION["cost_per_side_bps"],
)

second_day = momentum_backtest.step_portfolio_day(
    market_day=market.xs(second_date, level="Date"),
    units=first_day["units"],
    cash_gbp=first_day["cash_gbp"],
)

columns = [
    "equity_gbp",
    "cash_gbp",
    "total_cost_gbp",
    "dividends_gbp",
]

display(
    pd.DataFrame(
        [
            {column: result[column] for column in columns}
            for result in [first_day, second_day]
        ],
        index=pd.DatetimeIndex([first_date, second_date], name="Date"),
    ).round(2)
)

,equity_gbp,cash_gbp,total_cost_gbp,dividends_gbp
Date,,,,
2016-01-04,9810.01,0.0,9.99,0.0
2016-01-05,9732.94,0.0,0.00,0.0


In [14]:
importlib.reload(momentum_backtest)

development_backtest = momentum_backtest.run_momentum_backtest(
    market=market,
    targets=development_targets,
    sessions=development_dates,
    initial_capital_gbp=BASELINE["initial_capital_gbp"],
    cost_per_side_bps=EXECUTION["cost_per_side_bps"],
)

development_daily = development_backtest["daily"]
development_weights = development_backtest["weights"]
development_trades = development_backtest["trades"]

display(development_daily.head())

display(
    pd.Series({
        "Sessions": len(development_daily),
        "Scheduled rebalances": development_daily["is_rebalance"].sum(),
        "Final equity GBP": development_daily["equity_gbp"].iloc[-1],
        "Total trading costs GBP": development_daily["total_cost_gbp"].sum(),
        "Total dividends GBP": development_daily["dividends_gbp"].sum(),
    }).to_frame("Development")
)

,equity_gbp,net_return,cash_gbp,cash_weight,total_cost_gbp,dividends_gbp,traded_value_gbp,holdings_count,is_rebalance
Date,,,,,,,,,
2016-01-04,9810.012181,-0.018999,0.0,0.0,9.99001,0.0,9990.00999,11,True
2016-01-05,9732.935427,-0.007857,0.0,0.0,0.00000,0.0,0.00000,11,False
2016-01-06,9699.194580,-0.003467,0.0,0.0,0.00000,0.0,0.00000,11,False
2016-01-07,9620.338136,-0.008130,0.0,0.0,0.00000,0.0,0.00000,11,False
2016-01-08,9732.391609,0.011648,0.0,0.0,0.00000,0.0,0.00000,11,False


,Development
Sessions,1518.000000
Scheduled rebalances,72.000000
Final equity GBP,22841.337882
Total trading costs GBP,685.437354
Total dividends GBP,1154.492334


In [15]:
benchmark_targets, benchmark_selection = (
    momentum_backtest.build_momentum_targets(
        features,
        development_rebalances,
        top_frac=1.0,
    )
)

benchmark_backtest = momentum_backtest.run_momentum_backtest(
    market=market,
    targets=benchmark_targets,
    sessions=development_dates,
    initial_capital_gbp=BASELINE["initial_capital_gbp"],
    cost_per_side_bps=EXECUTION["cost_per_side_bps"],
)

benchmark_daily = benchmark_backtest["daily"]

development_equity = pd.DataFrame({
    "Momentum": development_daily["equity_gbp"],
    "Eligible-universe benchmark": benchmark_daily["equity_gbp"],
})

display(
    pd.DataFrame({
        name: {
            "Final equity GBP": daily["equity_gbp"].iloc[-1],
            "Trading costs GBP": daily["total_cost_gbp"].sum(),
            "Dividends GBP": daily["dividends_gbp"].sum(),
        }
        for name, daily in {
            "Momentum": development_daily,
            "Eligible-universe benchmark": benchmark_daily,
        }.items()
    }).round(2)
)

ValueError: 2017-07-03: Unusable open_gbp for required holdings: ['PREM.L']